# Análise exploratória | S_ALR_87013611

**Tabela:** `dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor`  
**Ambiente:** DEV  
**Objeto:** view (confirmar em DESCRIBE EXTENDED)  
**Esquema:** 17 colunas: `cod_ledger_fonte` (string), `cod_empresa` (string), `num_exercicio_fiscal` (string), `cod_periodo_fiscal` (string), `cod_conta_razao` (string), `cod_fornecedor` (string), `cod_debito_credito` (string), `cod_moeda_empresa` (string), `vl_saldo_periodo` (double), `qt_partidas` (bigint), `dt_base_calculo_vencimento` (string), `cod_conta_reconciliacao_fornecedor` (string), `cod_posicao_vdf` (string), `cod_posicao_vdf_pai` (string), `cod_grupo_conta` (string), `nm_conta_razao` (string), `dateingest` (date).  
**Clustering informado nos comentários:** `cod_empresa`, `num_exercicio_fiscal` (confirmar no metadado).  

**Como usar:** executar todas as células SQL, salvar HTML ou notebook executado e comparar com extração SAP no mesmo dia. Cada consulta é independente e somente leitura. `cod_empresa` é empresa, **não** centro; não existe `cod_centro` no DESCRIBE. Não aplicar automaticamente os centros 4128/4014 como empresas. A definição e filtros da view devem ser lidos na seção 1.

| Severidade | Alerta |
|---|---|
| 🔴 ALTA | Sem coluna de centro: conciliação por centro depende de mapeamento/filtro comprovado. |
| 🟡 MÉDIA | Período `000` é transporte, não movimento; separar de `001` a `016`. |
| 🟡 MÉDIA | `vl_saldo_periodo` é double; tolerância de 0,005 na conciliação. |
| 🟡 MÉDIA | Fornecedor e datas podem ficar vazios conforme o tipo de partida; conferir por segmento. |
| ⚪ BAIXA | `dateingest` registra data de carga, não instante do snapshot. |

**Índice:** 1 Metadados; 2 Volumetria; 3 Empresa; 3.1 Filtros 4128/4014; 4 Granularidade; 5 Duplicidade; 6 Preenchimento; 6.1 Sem fonte; 7 Cardinalidade; 8 Domínio; 9 Indicadores; 10 Numérico; 11 Totais; 12 Datas; 13 Códigos; 14 Checksum; 15 Amostra; 16 Distribuição; 17 Freshness; 18 Regras S_ALR_87013611; 19 Amostra ampliada; 20 Resumo.


## 1. Metadados

Leia o comentário e a definição da view para identificar os filtros da tela SAP; DETAIL/HISTORY não são executados em view.


In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor;

`DESCRIBE DETAIL` e `DESCRIBE HISTORY` não são executados: o objeto tem nome de view. Confirme o tipo no resultado acima.


## 2. Volumetria

Conta linhas, combinações da chave lógica, empresas, contas, fornecedores preenchidos e partidas somadas.


In [0]:
SELECT COUNT(*) AS linhas, COUNT(DISTINCT named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_periodo_fiscal', `cod_periodo_fiscal`, 'cod_conta_razao', `cod_conta_razao`, 'cod_fornecedor', `cod_fornecedor`, 'cod_debito_credito', `cod_debito_credito`, 'cod_moeda_empresa', `cod_moeda_empresa`)) AS chaves_logicas, COUNT(DISTINCT cod_empresa) AS empresas, COUNT(DISTINCT cod_conta_razao) AS contas, COUNT(DISTINCT CASE WHEN trim(cod_fornecedor) <> '' THEN cod_fornecedor END) AS fornecedores, SUM(qt_partidas) AS partidas_somadas FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor;

## 3. Distribuição por empresa

Não há centro no esquema. Distribui por empresa e marca volumes candidatos apenas como referência, sem equiparar empresa a centro.


In [0]:
SELECT cod_empresa, COUNT(*) AS linhas, SUM(qt_partidas) AS partidas, SUM(CASE WHEN cod_periodo_fiscal = '000' THEN vl_saldo_periodo ELSE 0 END) AS saldo_transporte, SUM(CASE WHEN cod_periodo_fiscal <> '000' THEN vl_saldo_periodo ELSE 0 END) AS saldo_periodico, CASE WHEN COUNT(*) BETWEEN 10000 AND 300000 THEN 'CANDIDATO A CENARIO POR EMPRESA; VALIDAR FILTRO SAP' ELSE 'VALIDAR VOLUME' END AS avaliacao FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY cod_empresa ORDER BY linhas DESC;

## 3.1. Verificação dos códigos 4128 e 4014

Somente verifica se esses valores ocorrem como empresa. Sua presença não prova equivalência a centro.


In [0]:
SELECT cod_empresa, COUNT(*) AS linhas, MIN(length(trim(cod_empresa))) AS tamanho_min, MAX(length(trim(cod_empresa))) AS tamanho_max FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor WHERE regexp_replace(trim(cod_empresa), '^0+', '') IN ('4128','4014') GROUP BY cod_empresa ORDER BY cod_empresa;

## 4. Granularidade

Compara 5 chaves candidatas; a chave real exige razão linhas/chaves = 1,0 e nenhuma duplicidade.


In [0]:
SELECT 'cod_ledger_fonte + cod_empresa + num_exercicio_fiscal' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`)) AS chaves, COUNT(*) / NULLIF(COUNT(DISTINCT named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`)), 0) AS linhas_por_chave FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor
UNION ALL
SELECT 'cod_ledger_fonte + cod_empresa + num_exercicio_fiscal + cod_periodo_fiscal + cod_conta_razao' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_periodo_fiscal', `cod_periodo_fiscal`, 'cod_conta_razao', `cod_conta_razao`)) AS chaves, COUNT(*) / NULLIF(COUNT(DISTINCT named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_periodo_fiscal', `cod_periodo_fiscal`, 'cod_conta_razao', `cod_conta_razao`)), 0) AS linhas_por_chave FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor
UNION ALL
SELECT 'cod_ledger_fonte + cod_empresa + num_exercicio_fiscal + cod_periodo_fiscal + cod_conta_razao + cod_fornecedor' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_periodo_fiscal', `cod_periodo_fiscal`, 'cod_conta_razao', `cod_conta_razao`, 'cod_fornecedor', `cod_fornecedor`)) AS chaves, COUNT(*) / NULLIF(COUNT(DISTINCT named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_periodo_fiscal', `cod_periodo_fiscal`, 'cod_conta_razao', `cod_conta_razao`, 'cod_fornecedor', `cod_fornecedor`)), 0) AS linhas_por_chave FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor
UNION ALL
SELECT 'cod_ledger_fonte + cod_empresa + num_exercicio_fiscal + cod_periodo_fiscal + cod_conta_razao + cod_fornecedor + cod_debito_credito' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_periodo_fiscal', `cod_periodo_fiscal`, 'cod_conta_razao', `cod_conta_razao`, 'cod_fornecedor', `cod_fornecedor`, 'cod_debito_credito', `cod_debito_credito`)) AS chaves, COUNT(*) / NULLIF(COUNT(DISTINCT named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_periodo_fiscal', `cod_periodo_fiscal`, 'cod_conta_razao', `cod_conta_razao`, 'cod_fornecedor', `cod_fornecedor`, 'cod_debito_credito', `cod_debito_credito`)), 0) AS linhas_por_chave FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor
UNION ALL
SELECT 'cod_ledger_fonte + cod_empresa + num_exercicio_fiscal + cod_periodo_fiscal + cod_conta_razao + cod_fornecedor + cod_debito_credito + cod_moeda_empresa' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_periodo_fiscal', `cod_periodo_fiscal`, 'cod_conta_razao', `cod_conta_razao`, 'cod_fornecedor', `cod_fornecedor`, 'cod_debito_credito', `cod_debito_credito`, 'cod_moeda_empresa', `cod_moeda_empresa`)) AS chaves, COUNT(*) / NULLIF(COUNT(DISTINCT named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_periodo_fiscal', `cod_periodo_fiscal`, 'cod_conta_razao', `cod_conta_razao`, 'cod_fornecedor', `cod_fornecedor`, 'cod_debito_credito', `cod_debito_credito`, 'cod_moeda_empresa', `cod_moeda_empresa`)), 0) AS linhas_por_chave FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor;

## 5. Duplicidade

Separa chaves repetidas com linhas idênticas daquelas com versões diferentes, sem inferir a causa.


In [0]:
WITH grupos AS (SELECT `cod_ledger_fonte`, `cod_empresa`, `num_exercicio_fiscal`, `cod_periodo_fiscal`, `cod_conta_razao`, `cod_fornecedor`, `cod_debito_credito`, `cod_moeda_empresa`, COUNT(*) AS linhas, COUNT(DISTINCT md5(to_json(named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_periodo_fiscal', `cod_periodo_fiscal`, 'cod_conta_razao', `cod_conta_razao`, 'cod_fornecedor', `cod_fornecedor`, 'cod_debito_credito', `cod_debito_credito`, 'cod_moeda_empresa', `cod_moeda_empresa`, 'vl_saldo_periodo', `vl_saldo_periodo`, 'qt_partidas', `qt_partidas`, 'dt_base_calculo_vencimento', `dt_base_calculo_vencimento`, 'cod_conta_reconciliacao_fornecedor', `cod_conta_reconciliacao_fornecedor`, 'cod_posicao_vdf', `cod_posicao_vdf`, 'cod_posicao_vdf_pai', `cod_posicao_vdf_pai`, 'cod_grupo_conta', `cod_grupo_conta`, 'nm_conta_razao', `nm_conta_razao`, 'dateingest', `dateingest`)))) AS versoes FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY `cod_ledger_fonte`, `cod_empresa`, `num_exercicio_fiscal`, `cod_periodo_fiscal`, `cod_conta_razao`, `cod_fornecedor`, `cod_debito_credito`, `cod_moeda_empresa` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COALESCE(SUM(linhas - 1),0) AS linhas_excedentes, COUNT_IF(versoes = 1) AS chaves_com_linhas_identicas, COUNT_IF(versoes > 1) AS chaves_com_versoes_diferentes FROM grupos;

In [0]:
WITH grupos AS (SELECT `cod_ledger_fonte`, `cod_empresa`, `num_exercicio_fiscal`, `cod_periodo_fiscal`, `cod_conta_razao`, `cod_fornecedor`, `cod_debito_credito`, `cod_moeda_empresa`, COUNT(*) AS linhas, COUNT(DISTINCT md5(to_json(named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_periodo_fiscal', `cod_periodo_fiscal`, 'cod_conta_razao', `cod_conta_razao`, 'cod_fornecedor', `cod_fornecedor`, 'cod_debito_credito', `cod_debito_credito`, 'cod_moeda_empresa', `cod_moeda_empresa`, 'vl_saldo_periodo', `vl_saldo_periodo`, 'qt_partidas', `qt_partidas`, 'dt_base_calculo_vencimento', `dt_base_calculo_vencimento`, 'cod_conta_reconciliacao_fornecedor', `cod_conta_reconciliacao_fornecedor`, 'cod_posicao_vdf', `cod_posicao_vdf`, 'cod_posicao_vdf_pai', `cod_posicao_vdf_pai`, 'cod_grupo_conta', `cod_grupo_conta`, 'nm_conta_razao', `nm_conta_razao`, 'dateingest', `dateingest`)))) AS versoes FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY `cod_ledger_fonte`, `cod_empresa`, `num_exercicio_fiscal`, `cod_periodo_fiscal`, `cod_conta_razao`, `cod_fornecedor`, `cod_debito_credito`, `cod_moeda_empresa` HAVING COUNT(*) > 1) SELECT `cod_ledger_fonte`, `cod_empresa`, `num_exercicio_fiscal`, `cod_periodo_fiscal`, `cod_conta_razao`, `cod_fornecedor`, `cod_debito_credito`, `cod_moeda_empresa`, linhas, versoes FROM grupos ORDER BY linhas DESC LIMIT 30;

## 6. Preenchimento de todas as colunas

Nulos, strings vazias/sentinelas, zeros e valores úteis em cada coluna; fornecedor vazio pode ser esperado.


In [0]:
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor), p AS (SELECT stack(17, 'cod_ledger_fonte', 'string', COUNT_IF(`cod_ledger_fonte` IS NULL), COUNT_IF(`cod_ledger_fonte` IS NOT NULL AND lower(trim(`cod_ledger_fonte`)) IN ('', 'null', 'nan', 'none', '#', '-')), COUNT_IF(`cod_ledger_fonte` IS NOT NULL AND trim(`cod_ledger_fonte`) RLIKE '^0+([.,]0+)?$'), 'cod_empresa', 'string', COUNT_IF(`cod_empresa` IS NULL), COUNT_IF(`cod_empresa` IS NOT NULL AND lower(trim(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-')), COUNT_IF(`cod_empresa` IS NOT NULL AND trim(`cod_empresa`) RLIKE '^0+([.,]0+)?$'), 'num_exercicio_fiscal', 'string', COUNT_IF(`num_exercicio_fiscal` IS NULL), COUNT_IF(`num_exercicio_fiscal` IS NOT NULL AND lower(trim(`num_exercicio_fiscal`)) IN ('', 'null', 'nan', 'none', '#', '-')), COUNT_IF(`num_exercicio_fiscal` IS NOT NULL AND trim(`num_exercicio_fiscal`) RLIKE '^0+([.,]0+)?$'), 'cod_periodo_fiscal', 'string', COUNT_IF(`cod_periodo_fiscal` IS NULL), COUNT_IF(`cod_periodo_fiscal` IS NOT NULL AND lower(trim(`cod_periodo_fiscal`)) IN ('', 'null', 'nan', 'none', '#', '-')), COUNT_IF(`cod_periodo_fiscal` IS NOT NULL AND trim(`cod_periodo_fiscal`) RLIKE '^0+([.,]0+)?$'), 'cod_conta_razao', 'string', COUNT_IF(`cod_conta_razao` IS NULL), COUNT_IF(`cod_conta_razao` IS NOT NULL AND lower(trim(`cod_conta_razao`)) IN ('', 'null', 'nan', 'none', '#', '-')), COUNT_IF(`cod_conta_razao` IS NOT NULL AND trim(`cod_conta_razao`) RLIKE '^0+([.,]0+)?$'), 'cod_fornecedor', 'string', COUNT_IF(`cod_fornecedor` IS NULL), COUNT_IF(`cod_fornecedor` IS NOT NULL AND lower(trim(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')), COUNT_IF(`cod_fornecedor` IS NOT NULL AND trim(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$'), 'cod_debito_credito', 'string', COUNT_IF(`cod_debito_credito` IS NULL), COUNT_IF(`cod_debito_credito` IS NOT NULL AND lower(trim(`cod_debito_credito`)) IN ('', 'null', 'nan', 'none', '#', '-')), COUNT_IF(`cod_debito_credito` IS NOT NULL AND trim(`cod_debito_credito`) RLIKE '^0+([.,]0+)?$'), 'cod_moeda_empresa', 'string', COUNT_IF(`cod_moeda_empresa` IS NULL), COUNT_IF(`cod_moeda_empresa` IS NOT NULL AND lower(trim(`cod_moeda_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-')), COUNT_IF(`cod_moeda_empresa` IS NOT NULL AND trim(`cod_moeda_empresa`) RLIKE '^0+([.,]0+)?$'), 'vl_saldo_periodo', 'double', COUNT_IF(`vl_saldo_periodo` IS NULL), CAST(0 AS BIGINT), COUNT_IF(`vl_saldo_periodo` = 0), 'qt_partidas', 'bigint', COUNT_IF(`qt_partidas` IS NULL), CAST(0 AS BIGINT), COUNT_IF(`qt_partidas` = 0), 'dt_base_calculo_vencimento', 'string', COUNT_IF(`dt_base_calculo_vencimento` IS NULL), COUNT_IF(`dt_base_calculo_vencimento` IS NOT NULL AND lower(trim(`dt_base_calculo_vencimento`)) IN ('', 'null', 'nan', 'none', '#', '-')), COUNT_IF(`dt_base_calculo_vencimento` IS NOT NULL AND trim(`dt_base_calculo_vencimento`) RLIKE '^0+([.,]0+)?$'), 'cod_conta_reconciliacao_fornecedor', 'string', COUNT_IF(`cod_conta_reconciliacao_fornecedor` IS NULL), COUNT_IF(`cod_conta_reconciliacao_fornecedor` IS NOT NULL AND lower(trim(`cod_conta_reconciliacao_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')), COUNT_IF(`cod_conta_reconciliacao_fornecedor` IS NOT NULL AND trim(`cod_conta_reconciliacao_fornecedor`) RLIKE '^0+([.,]0+)?$'), 'cod_posicao_vdf', 'string', COUNT_IF(`cod_posicao_vdf` IS NULL), COUNT_IF(`cod_posicao_vdf` IS NOT NULL AND lower(trim(`cod_posicao_vdf`)) IN ('', 'null', 'nan', 'none', '#', '-')), COUNT_IF(`cod_posicao_vdf` IS NOT NULL AND trim(`cod_posicao_vdf`) RLIKE '^0+([.,]0+)?$'), 'cod_posicao_vdf_pai', 'string', COUNT_IF(`cod_posicao_vdf_pai` IS NULL), COUNT_IF(`cod_posicao_vdf_pai` IS NOT NULL AND lower(trim(`cod_posicao_vdf_pai`)) IN ('', 'null', 'nan', 'none', '#', '-')), COUNT_IF(`cod_posicao_vdf_pai` IS NOT NULL AND trim(`cod_posicao_vdf_pai`) RLIKE '^0+([.,]0+)?$'), 'cod_grupo_conta', 'string', COUNT_IF(`cod_grupo_conta` IS NULL), COUNT_IF(`cod_grupo_conta` IS NOT NULL AND lower(trim(`cod_grupo_conta`)) IN ('', 'null', 'nan', 'none', '#', '-')), COUNT_IF(`cod_grupo_conta` IS NOT NULL AND trim(`cod_grupo_conta`) RLIKE '^0+([.,]0+)?$'), 'nm_conta_razao', 'string', COUNT_IF(`nm_conta_razao` IS NULL), COUNT_IF(`nm_conta_razao` IS NOT NULL AND lower(trim(`nm_conta_razao`)) IN ('', 'null', 'nan', 'none', '#', '-')), COUNT_IF(`nm_conta_razao` IS NOT NULL AND trim(`nm_conta_razao`) RLIKE '^0+([.,]0+)?$'), 'dateingest', 'date', COUNT_IF(`dateingest` IS NULL), CAST(0 AS BIGINT), CAST(0 AS BIGINT)) AS (coluna, tipo, nulos, vazios, zeros) FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor) SELECT p.coluna, p.tipo, t.total, p.nulos, p.vazios, p.zeros, t.total - p.nulos - p.vazios - p.zeros AS uteis, CASE WHEN t.total = 0 THEN 'SEM LINHAS' WHEN p.nulos = t.total THEN '100% NULO; INVESTIGAR FONTE' WHEN p.nulos + p.vazios = t.total THEN 'SEM VALOR UTIL; VALIDAR REGRA' ELSE 'PREENCHIDO' END AS veredito FROM p CROSS JOIN t ORDER BY p.coluna;

## 6.1. Colunas sem fonte

Os comentários recebidos não declaram coluna sem fonte. Lista campos opcionais por regra e mede vazios por segmento, sem chamar isso de erro.


In [0]:
SELECT CASE WHEN cod_fornecedor IS NULL OR trim(cod_fornecedor) = '' THEN 'SEM FORNECEDOR' ELSE 'COM FORNECEDOR' END AS segmento, COUNT(*) AS linhas, COUNT_IF(dt_base_calculo_vencimento IS NULL OR trim(dt_base_calculo_vencimento) = '') AS data_base_vazia, COUNT_IF(cod_conta_reconciliacao_fornecedor IS NULL OR trim(cod_conta_reconciliacao_fornecedor) = '') AS reconciliacao_vazia, COUNT_IF(cod_posicao_vdf IS NULL OR trim(cod_posicao_vdf) = '') AS posicao_vdf_vazia FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY 1;

## 7. Cardinalidade

Distingue constantes e dimensões de alta cardinalidade.


In [0]:
SELECT stack(17, 'cod_ledger_fonte', COUNT(DISTINCT `cod_ledger_fonte`), 'cod_empresa', COUNT(DISTINCT `cod_empresa`), 'num_exercicio_fiscal', COUNT(DISTINCT `num_exercicio_fiscal`), 'cod_periodo_fiscal', COUNT(DISTINCT `cod_periodo_fiscal`), 'cod_conta_razao', COUNT(DISTINCT `cod_conta_razao`), 'cod_fornecedor', COUNT(DISTINCT `cod_fornecedor`), 'cod_debito_credito', COUNT(DISTINCT `cod_debito_credito`), 'cod_moeda_empresa', COUNT(DISTINCT `cod_moeda_empresa`), 'vl_saldo_periodo', COUNT(DISTINCT `vl_saldo_periodo`), 'qt_partidas', COUNT(DISTINCT `qt_partidas`), 'dt_base_calculo_vencimento', COUNT(DISTINCT `dt_base_calculo_vencimento`), 'cod_conta_reconciliacao_fornecedor', COUNT(DISTINCT `cod_conta_reconciliacao_fornecedor`), 'cod_posicao_vdf', COUNT(DISTINCT `cod_posicao_vdf`), 'cod_posicao_vdf_pai', COUNT(DISTINCT `cod_posicao_vdf_pai`), 'cod_grupo_conta', COUNT(DISTINCT `cod_grupo_conta`), 'nm_conta_razao', COUNT(DISTINCT `nm_conta_razao`), 'dateingest', COUNT(DISTINCT `dateingest`)) AS (coluna, distintos) FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor;

## 8. Domínio

Mostra os 8 valores mais frequentes em cada dimensão categórica relevante.


In [0]:
WITH freq AS (SELECT 'cod_ledger_fonte' AS coluna, CAST(`cod_ledger_fonte` AS STRING) AS valor, COUNT(*) AS linhas FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY `cod_ledger_fonte` UNION ALL SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS linhas FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY `cod_empresa` UNION ALL SELECT 'num_exercicio_fiscal' AS coluna, CAST(`num_exercicio_fiscal` AS STRING) AS valor, COUNT(*) AS linhas FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY `num_exercicio_fiscal` UNION ALL SELECT 'cod_periodo_fiscal' AS coluna, CAST(`cod_periodo_fiscal` AS STRING) AS valor, COUNT(*) AS linhas FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY `cod_periodo_fiscal` UNION ALL SELECT 'cod_debito_credito' AS coluna, CAST(`cod_debito_credito` AS STRING) AS valor, COUNT(*) AS linhas FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY `cod_debito_credito` UNION ALL SELECT 'cod_moeda_empresa' AS coluna, CAST(`cod_moeda_empresa` AS STRING) AS valor, COUNT(*) AS linhas FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY `cod_moeda_empresa` UNION ALL SELECT 'cod_grupo_conta' AS coluna, CAST(`cod_grupo_conta` AS STRING) AS valor, COUNT(*) AS linhas FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY `cod_grupo_conta`), rankeado AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY coluna ORDER BY linhas DESC, valor) AS ordem FROM freq) SELECT coluna, valor, linhas FROM rankeado WHERE ordem <= 8 ORDER BY coluna, ordem;

## 9. Indicadores

Não há booleanas no DESCRIBE. Examina o indicador débito/crédito e seus valores vazios.


In [0]:
SELECT cod_debito_credito, COUNT(*) AS linhas, SUM(vl_saldo_periodo) AS saldo, SUM(qt_partidas) AS partidas FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY cod_debito_credito ORDER BY linhas DESC;

## 10. Perfil numérico

Verifica extremos, mediana aproximada, negativos e zeros para saldo e quantidade de partidas.


In [0]:
SELECT MIN(vl_saldo_periodo) AS saldo_min, MAX(vl_saldo_periodo) AS saldo_max, AVG(vl_saldo_periodo) AS saldo_medio, percentile_approx(vl_saldo_periodo, 0.5) AS saldo_mediano, COUNT_IF(vl_saldo_periodo < 0) AS saldos_negativos, COUNT_IF(vl_saldo_periodo = 0) AS saldos_zero, MIN(qt_partidas) AS partidas_min, MAX(qt_partidas) AS partidas_max, AVG(qt_partidas) AS partidas_media, percentile_approx(qt_partidas, 0.5) AS partidas_mediana, COUNT_IF(qt_partidas <= 0) AS partidas_nao_positivas FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor;

## 11. Totais para conciliação

Agrupa saldos e partidas por empresa, exercício, moeda e natureza do período. Não mistura transporte com movimento.


In [0]:
SELECT cod_empresa, num_exercicio_fiscal, cod_moeda_empresa, CASE WHEN cod_periodo_fiscal = '000' THEN 'TRANSPORTE' ELSE 'PERIODICO_OU_OUTRO' END AS natureza_periodo, COUNT(*) AS linhas, SUM(vl_saldo_periodo) AS saldo_double, SUM(CAST(vl_saldo_periodo AS DECIMAL(38,3))) AS saldo_decimal_arredondado, SUM(qt_partidas) AS partidas FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY cod_empresa, num_exercicio_fiscal, cod_moeda_empresa, CASE WHEN cod_periodo_fiscal = '000' THEN 'TRANSPORTE' ELSE 'PERIODICO_OU_OUTRO' END ORDER BY cod_empresa, num_exercicio_fiscal, cod_moeda_empresa, natureza_periodo;

## 12. Datas

Examina a data-base em string e a data de carga sem pressupor formato único.


In [0]:
SELECT CASE WHEN dt_base_calculo_vencimento IS NULL OR trim(dt_base_calculo_vencimento) = '' THEN 'VAZIA' WHEN dt_base_calculo_vencimento = '00000000' THEN '00000000' WHEN dt_base_calculo_vencimento = '9999-12-31' THEN '9999-12-31' WHEN dt_base_calculo_vencimento RLIKE '^[0-9]{8}$' THEN 'AAAAMMDD' WHEN dt_base_calculo_vencimento RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}$' THEN 'AAAA-MM-DD' ELSE 'OUTRO' END AS formato, COUNT(*) AS linhas, MIN(dt_base_calculo_vencimento) AS exemplo_min, MAX(dt_base_calculo_vencimento) AS exemplo_max FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY 1 ORDER BY linhas DESC;

In [0]:
SELECT MIN(dateingest) AS primeira_carga, MAX(dateingest) AS ultima_carga, COUNT(DISTINCT dateingest) AS dias_carga FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor;

## 13. Códigos

Mede comprimento e zeros à esquerda dos códigos SAP; procura colisões de fornecedor após normalização.


In [0]:
SELECT 'cod_fornecedor' AS campo, MIN(length(trim(cod_fornecedor))) AS tam_min, MAX(length(trim(cod_fornecedor))) AS tam_max, COUNT_IF(trim(cod_fornecedor) RLIKE '^0+[0-9]+$') AS com_zeros_esquerda, COUNT(DISTINCT cod_fornecedor) AS codigos_originais, COUNT(DISTINCT regexp_replace(trim(cod_fornecedor), '^0+', '')) AS codigos_normalizados FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor WHERE cod_fornecedor IS NOT NULL AND trim(cod_fornecedor) <> '' UNION ALL SELECT 'cod_conta_razao', MIN(length(trim(cod_conta_razao))), MAX(length(trim(cod_conta_razao))), COUNT_IF(trim(cod_conta_razao) RLIKE '^0+[0-9]+$'), COUNT(DISTINCT cod_conta_razao), COUNT(DISTINCT regexp_replace(trim(cod_conta_razao), '^0+', '')) FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor WHERE cod_conta_razao IS NOT NULL AND trim(cod_conta_razao) <> '';

## 14. Checksum

Agrupa o hash de todas as colunas para contar linhas integralmente idênticas.


In [0]:
WITH h AS (SELECT md5(to_json(named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_periodo_fiscal', `cod_periodo_fiscal`, 'cod_conta_razao', `cod_conta_razao`, 'cod_fornecedor', `cod_fornecedor`, 'cod_debito_credito', `cod_debito_credito`, 'cod_moeda_empresa', `cod_moeda_empresa`, 'vl_saldo_periodo', `vl_saldo_periodo`, 'qt_partidas', `qt_partidas`, 'dt_base_calculo_vencimento', `dt_base_calculo_vencimento`, 'cod_conta_reconciliacao_fornecedor', `cod_conta_reconciliacao_fornecedor`, 'cod_posicao_vdf', `cod_posicao_vdf`, 'cod_posicao_vdf_pai', `cod_posicao_vdf_pai`, 'cod_grupo_conta', `cod_grupo_conta`, 'nm_conta_razao', `nm_conta_razao`, 'dateingest', `dateingest`))) AS checksum FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor), g AS (SELECT checksum, COUNT(*) AS linhas FROM h GROUP BY checksum) SELECT COUNT(*) AS linhas_distintas, SUM(linhas) AS linhas_totais, SUM(CASE WHEN linhas > 1 THEN linhas - 1 ELSE 0 END) AS copias_identicas_excedentes FROM g;

## 15. Amostra

Exibe 20 linhas com colunas explícitas. O DESCRIBE não identifica campos pessoais; código de fornecedor não é nome nem documento.


In [0]:
SELECT `cod_ledger_fonte`, `cod_empresa`, `num_exercicio_fiscal`, `cod_periodo_fiscal`, `cod_conta_razao`, `cod_fornecedor`, `cod_debito_credito`, `cod_moeda_empresa`, `vl_saldo_periodo`, `qt_partidas`, `dt_base_calculo_vencimento`, `cod_conta_reconciliacao_fornecedor`, `cod_posicao_vdf`, `cod_posicao_vdf_pai`, `cod_grupo_conta`, `nm_conta_razao`, `dateingest` FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor ORDER BY cod_empresa, num_exercicio_fiscal, cod_periodo_fiscal, cod_conta_razao, cod_fornecedor LIMIT 20;

## 16. Distribuição interna

Mostra a composição por período e débito/crédito em cada empresa.


In [0]:
SELECT cod_empresa, num_exercicio_fiscal, cod_periodo_fiscal, cod_debito_credito, COUNT(*) AS linhas, SUM(qt_partidas) AS partidas, SUM(vl_saldo_periodo) AS saldo FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY cod_empresa, num_exercicio_fiscal, cod_periodo_fiscal, cod_debito_credito ORDER BY cod_empresa, num_exercicio_fiscal, cod_periodo_fiscal, cod_debito_credito;

## 17. Freshness

Verifica se há mais de uma data de carga e qual é o volume em cada data; não conclui histórico apenas pelo nome do campo.


In [0]:
SELECT dateingest, COUNT(*) AS linhas, COUNT(DISTINCT cod_empresa) AS empresas, SUM(qt_partidas) AS partidas FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY dateingest ORDER BY dateingest DESC;

## 18. Regras da S_ALR_87013611

Testa períodos permitidos, segregação do transporte, fornecedor sem partida e cobertura VDF. Regras da origem que exigem tabelas externas ficam para a conciliação SAP.


In [0]:
SELECT cod_periodo_fiscal, COUNT(*) AS linhas, SUM(vl_saldo_periodo) AS saldo, SUM(qt_partidas) AS partidas, CASE WHEN cod_periodo_fiscal = '000' THEN 'TRANSPORTE' WHEN cod_periodo_fiscal RLIKE '^0(0[1-9]|1[0-6])$' THEN 'PERIODO 001-016' ELSE 'FORA DO DOMINIO DECLARADO' END AS classificacao FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY cod_periodo_fiscal ORDER BY cod_periodo_fiscal;

In [0]:
SELECT cod_ledger_fonte, cod_moeda_empresa, COUNT(*) AS linhas, SUM(vl_saldo_periodo) AS saldo FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY cod_ledger_fonte, cod_moeda_empresa ORDER BY linhas DESC;

In [0]:
SELECT CASE WHEN cod_fornecedor IS NULL OR trim(cod_fornecedor) = '' THEN 'SEM FORNECEDOR' ELSE 'COM FORNECEDOR' END AS fornecedor, CASE WHEN cod_posicao_vdf IS NULL OR trim(cod_posicao_vdf) = '' THEN 'FORA DA VDF OU SEM POSICAO' ELSE 'COM POSICAO' END AS vdf, COUNT(*) AS linhas, COUNT_IF(cod_posicao_vdf_pai IS NULL OR trim(cod_posicao_vdf_pai) = '') AS pai_vazio, COUNT_IF(cod_conta_reconciliacao_fornecedor IS NULL OR trim(cod_conta_reconciliacao_fornecedor) = '') AS reconciliacao_vazia, COUNT_IF(dt_base_calculo_vencimento IS NULL OR trim(dt_base_calculo_vencimento) = '') AS data_base_vazia FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY 1, 2 ORDER BY fornecedor, vdf;

In [0]:
SELECT cod_empresa, num_exercicio_fiscal, COUNT(*) AS linhas, SUM(CASE WHEN cod_periodo_fiscal = '000' THEN vl_saldo_periodo ELSE 0 END) AS transporte, SUM(CASE WHEN cod_periodo_fiscal BETWEEN '001' AND '016' THEN vl_saldo_periodo ELSE 0 END) AS periodos_001_016, SUM(CASE WHEN cod_periodo_fiscal NOT BETWEEN '000' AND '016' OR cod_periodo_fiscal IS NULL THEN vl_saldo_periodo ELSE 0 END) AS outros_periodos FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor GROUP BY cod_empresa, num_exercicio_fiscal ORDER BY cod_empresa, num_exercicio_fiscal;

**Limites de validação:** a menor data por grupo, a correspondência da partida contábil, o cadastro de fornecedor, a hierarquia SAP e a igualdade com a origem não podem ser confirmados apenas com esta view. Confronte com as fontes SAP no mesmo recorte.


## 19. Amostra ampliada

Exibe até 500 linhas para inspeção, com seleção explícita de campos.


In [0]:
SELECT `cod_ledger_fonte`, `cod_empresa`, `num_exercicio_fiscal`, `cod_periodo_fiscal`, `cod_conta_razao`, `cod_fornecedor`, `cod_debito_credito`, `cod_moeda_empresa`, `vl_saldo_periodo`, `qt_partidas`, `dt_base_calculo_vencimento`, `cod_conta_reconciliacao_fornecedor`, `cod_posicao_vdf`, `cod_posicao_vdf_pai`, `cod_grupo_conta`, `nm_conta_razao`, `dateingest` FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor WHERE cod_empresa = 4014 ORDER BY cod_empresa, num_exercicio_fiscal, cod_periodo_fiscal, cod_conta_razao, cod_fornecedor;

## 20. Resumo

Copie os resultados para planejar cenários; a chave declarada só é confirmada quando linhas = chaves.


In [0]:
SELECT COUNT(*) AS linhas, COUNT(DISTINCT named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_periodo_fiscal', `cod_periodo_fiscal`, 'cod_conta_razao', `cod_conta_razao`, 'cod_fornecedor', `cod_fornecedor`, 'cod_debito_credito', `cod_debito_credito`, 'cod_moeda_empresa', `cod_moeda_empresa`)) AS chaves_logicas, COUNT(*) - COUNT(DISTINCT named_struct('cod_ledger_fonte', `cod_ledger_fonte`, 'cod_empresa', `cod_empresa`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_periodo_fiscal', `cod_periodo_fiscal`, 'cod_conta_razao', `cod_conta_razao`, 'cod_fornecedor', `cod_fornecedor`, 'cod_debito_credito', `cod_debito_credito`, 'cod_moeda_empresa', `cod_moeda_empresa`)) AS linhas_excedentes_chave, COUNT(DISTINCT cod_empresa) AS empresas, COUNT_IF(cod_periodo_fiscal = '000') AS linhas_transporte, COUNT_IF(cod_periodo_fiscal BETWEEN '001' AND '016') AS linhas_periodicas, SUM(qt_partidas) AS partidas, MIN(dateingest) AS carga_min, MAX(dateingest) AS carga_max FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor;

### Checklist para comparação com SAP

- [ ] Confirmar no `DESCRIBE EXTENDED` o tipo, comentário e filtros aplicados pela view.
- [ ] Confirmar se há mapeamento comprovado entre centro SAP e empresa; caso contrário, comparar por empresa/exercício/conta/período.
- [ ] Extrair SAP e Datalake no mesmo dia e registrar os filtros de empresa, exercício, ledger, moeda e período.
- [ ] Comparar transporte `000` separadamente dos períodos `001` a `016`.
- [ ] Verificar chave real, duplicatas idênticas e versões diferentes antes de gerar cenários.
- [ ] Normalizar zeros à esquerda e formatos de data nos dois lados; para `double`, usar tolerância de 0,005.
- [ ] Não tratar fornecedor ausente como defeito sem validar a natureza da partida.
- [ ] Enviar HTML ou `.ipynb` executado para definir volume esperado e filtros dos cenários.


In [0]:
WITH valores AS (
  SELECT stack(14,
    'cod_ledger_fonte', cod_ledger_fonte,
    'cod_empresa', cod_empresa,
    'num_exercicio_fiscal', num_exercicio_fiscal,
    'cod_periodo_fiscal', cod_periodo_fiscal,
    'cod_conta_razao', cod_conta_razao,
    'cod_fornecedor', cod_fornecedor,
    'cod_debito_credito', cod_debito_credito,
    'cod_moeda_empresa', cod_moeda_empresa,
    'dt_base_calculo_vencimento', dt_base_calculo_vencimento,
    'cod_conta_reconciliacao_fornecedor', cod_conta_reconciliacao_fornecedor,
    'cod_posicao_vdf', cod_posicao_vdf,
    'cod_posicao_vdf_pai', cod_posicao_vdf_pai,
    'cod_grupo_conta', cod_grupo_conta,
    'nm_conta_razao', nm_conta_razao
  ) AS (coluna, valor)
  FROM dev_procurement.corp_curated.vw_ds_ctrl_s_alr_87013611_saldos_conta_fornecedor
)
SELECT
  coluna,
  COUNT_IF(trim(valor) = '4128') AS correspondencias_exatas,
  COUNT_IF(
    trim(valor) <> '4128'
    AND regexp_replace(trim(valor), '^0+', '') = '4128'
  ) AS correspondencias_com_zeros_esquerda,
  COUNT_IF(
    valor LIKE '%4128%'
    AND regexp_replace(trim(valor), '^0+', '') <> '4128'
  ) AS correspondencias_parciais,
  MIN(CASE WHEN valor LIKE '%4128%' THEN valor END) AS exemplo
FROM valores
WHERE valor LIKE '%4128%'
GROUP BY coluna
ORDER BY coluna;